# Munti — full training run (free Kaggle T4)

This notebook holds **no logic**. Everything it calls lives in the `munti/`
package in the repo, which is committed and tested — the notebook only wires
up paths and presses go. That keeps the run reproducible (NFR-3) and stops the
notebook and the repo from drifting apart.

## Setup (once)

1. Zip the repo and upload it as a Kaggle **Dataset** named `munti`.
2. New Notebook → *Add Data* → your `munti` dataset.
3. **Settings → Accelerator → GPU T4 x2** (we use one) and *Internet → On*
   (needed to download TinyStories).
4. Run all.
5. **Save Version → Save & Run All** so `/kaggle/working` persists.

## If the session dies (9h limit)

Start a new notebook, and in *Add Data* attach **this notebook's output** as
well as the `munti` dataset. Cell 2 finds the old checkpoint automatically and
training resumes at the exact step — optimizer state included, so it's a true
continuation, not a warm restart.

In [ ]:
# --- 1. locate the repo, make a writable copy, import the package ---
import os, shutil, subprocess, sys, glob
from pathlib import Path

SRC = Path("/kaggle/input/munti")          # the uploaded repo (read-only)
WORK = Path("/kaggle/working/munti-repo")  # writable copy: data/ and out/ are written here

if not WORK.exists():
    # The zip may or may not have nested the repo in a folder; find the real root.
    root = next((p.parent for p in SRC.rglob("pyproject.toml")), SRC)
    shutil.copytree(root, WORK)
os.chdir(WORK)
sys.path.insert(0, str(WORK))

import torch
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — turn on the accelerator")
print("bf16:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else "n/a", "(T4 is fp16-only; the loop handles it)")
print("repo:", WORK, sorted(p.name for p in WORK.iterdir()))

In [ ]:
# --- 2. restore a previous session's checkpoint, if one was attached ---
# Looks through every attached dataset for a prior run's out/ directory.
RESUME = False
for ckpt in glob.glob("/kaggle/input/*/**/out/ckpt.pt", recursive=True):
    prev = Path(ckpt).parent
    shutil.copytree(prev, WORK / "out", dirs_exist_ok=True)
    # The tokenizer must be the one that produced those weights, not a new one.
    tok = Path(ckpt).parent.parent / "data" / "tokenizer.json"
    if tok.exists():
        (WORK / "data").mkdir(exist_ok=True)
        shutil.copy(tok, WORK / "data" / "tokenizer.json")
    RESUME = True
    print("resuming from", ckpt)
    break
else:
    print("fresh run")

In [ ]:
# --- 3. correctness gate ---
# ~40s on CPU, and it has already passed locally. Re-running it here proves the
# code that actually got uploaded is the code that passed, not a stale zip.
print(subprocess.run([sys.executable, "test_munti.py"], capture_output=True, text=True).stdout)

In [ ]:
# --- 4. data: download TinyStories, train the BPE, write the token streams ---
# A few minutes. Skipped if a previous session already built them.
from munti import data as D

if not (WORK / "data" / "train.bin").exists():
    D.prepare(limit=None, vocab_size=4096)   # limit=None => the full 2.1M stories
else:
    print("token streams already present, skipping")
print("train tokens:", f"{len(D.load_split('train')):,}")

In [ ]:
# --- 5. train ---
# ~12.5M params, 20k steps at batch 64 x 256 tokens (~330M tokens seen).
# Checkpoints + probe samples land in out/ every 500 steps.
from munti.train import train

train("configs/munti-12m.yaml", resume=RESUME)

In [ ]:
# --- 6. artifacts: loss curve + generations from held-out prompts ---
from munti.train import plot_curve
from munti.model import Munti
from munti.sample import generate_text
from munti import tokenizer as tk

plot_curve("out/loss.csv")

model = Munti.from_checkpoint(torch.load("out/ckpt.pt", map_location="cuda", weights_only=False), device="cuda")
tok = tk.load()
for prompt in [
    "Once upon a time, there was a little girl named Lily.",
    "Tom found a shiny red box under the tree. He",
    "The cat was hungry, so",
]:
    print("-" * 70)
    print(generate_text(model, tok, prompt, device="cuda", max_new_tokens=200, temperature=0.8, top_k=200))

In [ ]:
# --- 7. things a 12M model should fail at (for the honest can-do/can't-do section) ---
# Not a formality: these outputs go in the README verbatim. TinyStories teaches
# simple narrative prose and nothing else, so questions, facts and dialogue
# should all come back as story-shaped noise.
for prompt in [
    "What is the capital of France?",
    "Q: How many legs does a spider have? A:",
    "def fibonacci(n):",
    "The mitochondria is",
]:
    print("-" * 70)
    print(generate_text(model, tok, prompt, device="cuda", max_new_tokens=80, temperature=0.8, top_k=200))

In [ ]:
# --- 8. bundle the artifacts to download ---
# Commit these to the repo: loss.csv, curve.png, samples.md, tokenizer.json
# (force-add the tokenizer — it's gitignored so a dev-slice one can't sneak in).
shutil.copy("data/tokenizer.json", "out/tokenizer.json")
shutil.make_archive("/kaggle/working/munti-artifacts", "zip", "out")
print(sorted(os.listdir("out")))